# Load Model and its Tokenizer

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", device_map="auto")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

# Load Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")
dataset

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

# Tokenization for inputs(dialogue) and targets(summary)

In [3]:
def tokenize_data(batch):

    inputs = tokenizer(
        batch["dialogue"],
        max_length=1024,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

# apply the function on all data

tokenized_dataset = dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

# Data Collator and TrainingArguments

In [4]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./distilbart-samsum",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True
)

# Training

In [5]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.526112,1.426390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3683, training_loss=1.6540115360588346, metrics={'train_runtime': 1721.7863, 'train_samples_per_second': 8.556, 'train_steps_per_second': 2.139, 'total_flos': 5979079500914688.0, 'train_loss': 1.6540115360588346, 'epoch': 1.0})

In [6]:
# test the model

import torch
test_text = dataset["test"][0]["dialogue"]

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(model.device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=60,
        min_length=10,
        num_beams=4
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("MODEL SUMMARY:")
print(summary)

print("\nREFERENCE SUMMARY:")
print(dataset["test"][0]["summary"])

MODEL SUMMARY:
Amanda can't find Betty's number. Larry called her last time they were at the park together.

REFERENCE SUMMARY:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [7]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


# Evaluate

In [8]:
import evaluate
rouge = evaluate.load("rouge")

test_dataset = tokenized_dataset["test"]
print(len(test_dataset))

import torch

model.eval()

predictions = []

batch_size = 8

for i in range(0, len(test_dataset), batch_size):

    batch = test_dataset[i:i + batch_size]

    inputs = tokenizer.pad(
        {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        },
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        summary_ids = model.generate(
            **inputs,
            max_length=60,
            min_length=10,
            num_beams=4,
            early_stopping=True
        )

    summaries = tokenizer.batch_decode(
        summary_ids,
        skip_special_tokens=True
    )

    predictions.extend(summaries)

print("Number of predictions:", len(predictions))

819
Number of predictions: 819


In [9]:
# check the total references number
references = dataset["test"]["summary"]
print("Number of references:", len(references))



Number of references: 819


# Compute result

In [10]:
results = rouge.compute(
    predictions=predictions,
    references=references
)

results


{'rouge1': np.float64(0.5120365924979668),
 'rouge2': np.float64(0.2659948315536548),
 'rougeL': np.float64(0.4220377242425455),
 'rougeLsum': np.float64(0.4223350702185948)}

In [11]:
# convert results into %

for metric, score in results.items():
    print(f"{metric}: {score * 100:.2f}%")

rouge1: 51.20%
rouge2: 26.60%
rougeL: 42.20%
rougeLsum: 42.23%


# Save Model and Tokenizer

In [12]:
save_path = "./distilbart-samsum"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbart-samsum/tokenizer_config.json',
 './distilbart-samsum/tokenizer.json')

# Test Saving

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_test = AutoTokenizer.from_pretrained(save_path)

model_test = AutoModelForSeq2SeqLM.from_pretrained(
    save_path
)

text = dataset["test"][0]["dialogue"]

inputs = tokenizer_test(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(model_test.device)

summary_ids = model_test.generate(
    **inputs,
    max_length=60,
    min_length=10,
    num_beams=4
)

summary = tokenizer_test.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

Loading weights:   0%|          | 0/356 [00:00<?, ?it/s]

Amanda can't find Betty's number. Larry called her last time they were at the park together.


# Push the Model to Hugging Face

In [ ]:
!zip -r distilbart-samsum.zip ./distilbart-samsum

  adding: distilbart-samsum/ (stored 0%)
  adding: distilbart-samsum/config.json (deflated 61%)
  adding: distilbart-samsum/generation_config.json (deflated 47%)
  adding: distilbart-samsum/checkpoint-3683/ (stored 0%)
  adding: distilbart-samsum/checkpoint-3683/config.json (deflated 61%)
  adding: distilbart-samsum/checkpoint-3683/rng_state.pth (deflated 26%)
  adding: distilbart-samsum/checkpoint-3683/trainer_state.json (deflated 71%)
  adding: distilbart-samsum/checkpoint-3683/scheduler.pt (deflated 62%)
  adding: distilbart-samsum/checkpoint-3683/generation_config.json (deflated 47%)
  adding: distilbart-samsum/checkpoint-3683/tokenizer.json (deflated 82%)
  adding: distilbart-samsum/checkpoint-3683/model.safetensors (deflated 7%)
  adding: distilbart-samsum/checkpoint-3683/optimizer.pt (deflated 8%)
  adding: distilbart-samsum/checkpoint-3683/training_args.bin (deflated 53%)
  adding: distilbart-samsum/checkpoint-3683/tokenizer_config.json (deflated 50%)
  adding: distilbart-samsu

In [ ]:
from google.colab import files
files.download("distilbart-samsum.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls -lh distilbart-samsum.zip

-rw-r--r-- 1 root root 4.2G Oct  3 22:33 distilbart-samsum.zip


In [15]:
!mkdir -p distilbart-samsum-clean

!cp distilbart-samsum/config.json distilbart-samsum-clean/
!cp distilbart-samsum/generation_config.json distilbart-samsum-clean/
!cp distilbart-samsum/model.safetensors distilbart-samsum-clean/
!cp distilbart-samsum/tokenizer.json distilbart-samsum-clean/
!cp distilbart-samsum/tokenizer_config.json distilbart-samsum-clean/

In [16]:
!du -sh distilbart-samsum-clean

1.2G	distilbart-samsum-clean


In [17]:
!zip -r distilbart-samsum-clean.zip ./distilbart-samsum-clean

  adding: distilbart-samsum-clean/ (stored 0%)
  adding: distilbart-samsum-clean/config.json (deflated 61%)
  adding: distilbart-samsum-clean/generation_config.json (deflated 47%)
  adding: distilbart-samsum-clean/tokenizer.json (deflated 82%)
  adding: distilbart-samsum-clean/model.safetensors (deflated 7%)
  adding: distilbart-samsum-clean/tokenizer_config.json (deflated 50%)


In [18]:
!ls -lh distilbart-samsum-clean.zip

-rw-r--r-- 1 root root 1.1G Oct  4 14:08 distilbart-samsum-clean.zip


In [19]:
!pip install -q huggingface_hub

In [20]:
from huggingface_hub import login

login()

In [21]:
from huggingface_hub import create_repo

create_repo(
    "distilbart-samsum",
    repo_type="model"
)

RepoUrl('https://huggingface.co/Ahmed-1511/distilbart-samsum', endpoint='https://huggingface.co', repo_type='model', repo_id='Ahmed-1511/distilbart-samsum')

In [22]:
from huggingface_hub import upload_folder

upload_folder(
    repo_id="Ahmed-1511/distilbart-samsum",
    folder_path="./distilbart-samsum-clean",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/Ahmed-1511/distilbart-samsum/commit/67b4ce46d312ddfad943c4ccd2396e0cdfb667be', commit_message='Upload folder using huggingface_hub', commit_description='', oid='67b4ce46d312ddfad943c4ccd2396e0cdfb667be', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ahmed-1511/distilbart-samsum', endpoint='https://huggingface.co', repo_type='model', repo_id='Ahmed-1511/distilbart-samsum'), pr_revision=None, pr_num=None)